# 05 · Pressures, capillary pressure, saturation–height and uncertainty
**F-O gas field — Notebook 05 (Paper 2 core analysis)**

Inputs: unzipped **F-O_composites_v5**; `pressures_all.csv` (every RFT/XPT pretest decoded from the tapes, with quality flag);
`pc_all.csv` (37 capillary-pressure curves: F-O6 MICP ×9, F-O6 porous plate ×9, F-O2 porous plate ×18-19);
`pressure_qc_traces.png` (every pressure trace with its pick — check any pressure by eye).

Questions
1. Are the wells in pressure communication, and can pressures fix the free-water level (FWL)?
2. Does core capillary pressure explain the high-Sw sands above the log GWC (capillary transition) — or not (perched water / compartments)?
3. Which log-Sw model does the independent core evidence support (SCAL m* vs water-leg m* from Notebook 04)?
4. What is the uncertainty range of hydrocarbon pore thickness and net pay per well?

In [ ]:
# Cell 1 — paths (EDIT)
from pathlib import Path
V5_DIR   = Path(r"C:\Users\YOUR_NAME\...\F-O_composites_v5")
DATA_DIR = Path(r"C:\Users\YOUR_NAME\...\FO_petro_v5")        # folder with pressures_all.csv and pc_all.csv
OUT_DIR  = Path(r"C:\Users\YOUR_NAME\...\F-O_nb05_outputs")
OUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
# Cell 2 — load
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from petrotools import shm, saturation as st
WELLS = ["F-O1", "F-O2", "F-O3", "F-O4", "F-O6", "F-O8", "F-S1"]
logs = {w: pd.read_csv(V5_DIR / f"{w}_composite_v5.csv", index_col=0) for w in WELLS}
P = pd.read_csv(DATA_DIR / "pressures_all.csv"); PC = pd.read_csv(DATA_DIR / "pc_all.csv")
P["tvdss"] = P.depth_md - P.well.map(shm.KB)
P.groupby(["well", "quality"]).size().unstack(fill_value=0)

## Step 1 — formation pressures
Each pretest was decoded from the raw tapes (F-O1 LIS strain gauge; F-O3 LIS quartz; F-O4 STA DLIS; F-O8 XPT DLIS). The
formation pressure is the median of the longest stable part of the build-up (|dp/dt| < 0.05 psi/s for ≥ 15 s). Most tests
are **tight/dry** (no build-up) — itself evidence of the < 0.1 mD background sand — and several build up to mud pressure
(**supercharged**). Reference: the F-O2 DST, 7721.5 psia at 3619 m TVDSS, and the PVT gas gradient 0.325 psi/m (0.0991 psi/ft).

In [ ]:
# Cell 3
P["resid_vs_gas_line_psi"] = shm.pressure_vs_gas_line(P.tvdss, P.pform_psia)
# plausibility: a static formation pressure cannot lie far BELOW the gas line of a connected column -> unfinished build-up
P.loc[(P.quality == "good") & (P.resid_vs_gas_line_psi < -300), "quality"] = "implausible (far below gas line: unfinished build-up)"
good = P[P.quality == "good"].copy()
f, ax = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
for w, g in good.groupby("well"): ax[0].scatter(g.pform_psia, g.tvdss, s=22, label=w)
z = np.linspace(3580, 3880, 50); ax[0].plot(7721.5 + 0.325 * (z - 3619), z, "k-", lw=1, label="gas line (F-O2 DST)")
ax[0].scatter([7721.5], [3619], marker="*", s=150, color="k", label="F-O2 DST")
ax[0].axhline(shm.GWC_TVDSS, color="#eb6834", lw=0.8); ax[0].invert_yaxis(); ax[0].set_xlabel("pressure (psia)"); ax[0].set_ylabel("TVDSS (m)")
ax[0].set_xlim(7600, 8100); ax[0].legend(fontsize=7)
for w, g in good.groupby("well"): ax[1].scatter(g.resid_vs_gas_line_psi, g.tvdss, s=22)
ax[1].axvline(0, color="k", lw=0.8); ax[1].axhline(shm.GWC_TVDSS, color="#eb6834", lw=0.8); ax[1].set_xlabel("residual to gas line (psi)")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_pressure_vs_depth.png", dpi=150); plt.show()
good.groupby("well").resid_vs_gas_line_psi.agg(["count", "min", "median"]).round(1)

**Reading it.** The lowest good pressures in **F-O1 (3691–3700 m), F-O8 (3683–3705 m) and the F-O2 DST lie on one gas
gradient within ~2–10 psi** → the gas leg is in pressure communication across the crest. F-O4 (+35 to +200 psi) and F-O3
(+400 psi) read high: these are the tightest test intervals, and high readings with build-ups to near mud pressure are the
signature of supercharging (as the F-O4 operator concluded). Water-leg tests scatter by ~±70 psi, so **pressures cannot fix the
FWL**; the field is strongly overpressured (~7750 psia at 3700 m vs ~5450 psi normal hydrostatic).

## Step 2 — capillary pressure → Leverett J
Lab data are converted with σcosθ = 72 (air–brine), 371.5 (air–Hg, 485 dyn/cm, 140°) and 35 dyn/cm for gas–brine at reservoir
conditions; J = 0.2166·Pc·√(k/φ)/(σcosθ). The **F-O6 porous-plate** curves are excluded from the fit: on 0.01–0.07 mD plugs
the 1–60 psi steps do not reach equilibrium (flat curves, Sw 0.6–0.85 at 60 psi), whereas MICP on the same depths does.

In [ ]:
# Cell 4
fitset = PC[~((PC.well == "F-O6") & (PC.method == "porous plate"))]
A, B, SD = shm.fit_J(fitset)
f, a = plt.subplots(figsize=(6.5, 5))
for (w, mth), g in PC.groupby(["well", "method"]):
    g = g[(g.sw < 0.99) & (g.J > 0)]; a.scatter(g.sw, g.J, s=4, alpha=0.4, label=f"{w} {mth}")
s = np.linspace(0.05, 1, 100); a.plot(s, (s / A) ** (1 / B), "k-", lw=1.5, label=f"fit Sw = {A:.3f} J^{B:.3f}")
a.set_yscale("log"); a.set_xlim(0, 1); a.set_ylim(1e-3, 100); a.set_xlabel("Sw"); a.set_ylabel("Leverett J"); a.legend(fontsize=7); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_J_function.png", dpi=150); plt.show()
print(f"J fit (F-O2 porous plate + F-O6 MICP): Sw = {A:.3f} J^{B:.3f}; scatter {SD:.2f} dex")

## Step 3 — saturation–height model vs log Sw
With the FWL at the log GWC (3797 m TVDSS; 3812 m as a check for a threshold height), Sw is predicted at every log sample from
height, PHIT and PERM, and compared with the log Sw models in the gas leg (clean sand, PHIT > 5 %, above the GWC minus 5 m).

In [ ]:
# Cell 5
rows = []
for w in WELLS:
    d = logs[w]; t, b = shm.WIN[w]; s = d.loc[t:b]; z = s.index.to_numpy() - shm.KB[w]
    m = (z < shm.GWC_TVDSS - 5) & (s.PHIT > 0.05) & (s.VSH_GR < 0.4) & s.SW.notna() & s.PERM.notna(); s, z = s[m], z[m]
    for fwl in (shm.GWC_TVDSS, shm.GWC_TVDSS + 15):
        pred = shm.sw_shm(fwl - z, s.PERM.to_numpy(), s.PHIT.to_numpy(), A, B)
        rows.append(dict(well=w, fwl_tvdss=fwl, n=len(s), Sw_core_Pc=np.median(pred), Sw_log_SCALm=s.SW_WS_SCAL.median(),
                         Sw_log_waterleg_m=s.SW.median(), mae_vs_SCALm=np.median(np.abs(pred - s.SW_WS_SCAL)),
                         mae_vs_waterleg_m=np.median(np.abs(pred - s.SW))))
cmp = pd.DataFrame(rows).round(2); cmp.to_csv(OUT_DIR / "shm_vs_log_sw.csv", index=False); cmp

In [ ]:
# Cell 6 — profiles
f, ax = plt.subplots(1, len(WELLS), figsize=(17, 6), sharey=True)
for a, w in zip(ax, WELLS):
    d = logs[w]; t, b = shm.WIN[w]; s = d.loc[t - 20:b + 60]; s = s[(s.PHIT > 0.05) & (s.VSH_GR < 0.4)]; z = s.index.to_numpy() - shm.KB[w]
    a.scatter(s.SW, z, s=2, alpha=0.35, color="0.6", label="log, water-leg m* (NB04 base)")
    a.scatter(s.SW_WS_SCAL, z, s=2, alpha=0.5, color="#2a78d6", label="log, SCAL m*")
    a.scatter(shm.sw_shm(shm.GWC_TVDSS - z, s.PERM.to_numpy(), s.PHIT.to_numpy(), A, B), z, s=2, alpha=0.6, color="#eb6834", label="core Pc model (FWL at GWC)")
    a.axhline(shm.GWC_TVDSS, color="k", lw=0.8); a.set_xlim(0, 1); a.set_title(w, fontsize=9); a.grid(alpha=0.3); a.set_xlabel("Sw", fontsize=8)
ax[0].set_ylim(3880, 3580); ax[0].set_ylabel("TVDSS (m)"); ax[0].legend(fontsize=6.5, markerscale=4, loc="lower left")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_shm_vs_log_profiles.png", dpi=150); plt.show()

**What the core evidence says**
* In **F-O3, F-O4 and F-O6** the capillary model follows the **SCAL-parameter** log Sw closely, including F-O6's gradual
  increase down to the contact — a capillary transition zone tens of metres thick, as expected for 0.01–1 mD rock.
* In F-O1 and F-O2 the capillary model is lower still (better rock, steeper J-curve).
* The **water-leg m*** of Notebook 04 is 0.15–0.35 more pessimistic than the core model everywhere. The two readings are:
  (a) SCAL parameters hold, and the Sw ≈ 0.7 below the present GWC is **residual gas** (a paleo-column, later leakage/tilt);
  (b) the water-leg m* holds and the lab Pc (ambient stress) is optimistic. Both are carried as end-members below.
* **F-S1** (and partly F-O8) keeps high log Sw even with SCAL parameters, far above its capillary prediction: the high-Sw
  sands there are **not** a capillary transition — candidates are perched water, a separate compartment or a deeper/different
  contact. This is the anomaly worth pursuing (Paper 2).

## Step 4 — Monte Carlo: hydrocarbon pore thickness (HCPV = Σ φ(1−Sw)h) and net pay per well

In [ ]:
# Cell 7
PHI_CUT = {"F-O1": 0.063, "F-O2": 0.096, "F-O3": 0.093, "F-O4": 0.065, "F-O6": 0.094, "F-S1": 0.098}; PHI_FIELD = 0.087
N, rng = 1000, np.random.default_rng(42)
mc_rows, draws = [], {}
for w in WELLS:
    d = logs[w]; top, base = shm.WIN[w]; base = min(base, shm.GWC_TVDSS + shm.KB[w]); s = d.loc[top:base]
    s = s[s.PHIT.notna() & s[st.RT_CURVE[w]].notna() & (s.VSH_GR <= 0.4)]
    rt, phi0, rw0, Bc = s[st.RT_CURVE[w]].to_numpy(), s.PHIT.to_numpy(), s.RW.to_numpy(), s.B_WS.to_numpy()
    m = rng.uniform(2.10, 2.44, N)        # SCAL m* <-> water-leg m*
    n = rng.uniform(1.91, 2.88, N)        # F-O2 <-> F-O6 n*
    rwf = rng.normal(1.0, 0.10, N)        # Rw +/-10 %
    dphi = rng.normal(0.0, 0.01, N)       # systematic porosity calibration +/-1 p.u.
    hc, npay = np.empty(N), np.empty(N)
    for i in range(N):
        phi = np.clip(phi0 + dphi[i], 0.005, 0.35); qv = np.clip(st.qv_from_cec(1.0, phi), 0, 5)
        sw = st.waxman_smits_sw(rt, phi, rw0 * rwf[i], qv, Bc, m[i], n[i], iters=30)
        pay = (phi >= PHI_CUT.get(w, PHI_FIELD)) & (sw <= 0.65)
        hc[i] = (phi * (1 - sw))[pay].sum() * 0.1524; npay[i] = pay.sum() * 0.1524
    draws[w] = pd.DataFrame(dict(m=m, n=n, rwf=rwf, dphi=dphi, hcpv=hc, netpay=npay))
    q = lambda x, p: float(np.percentile(x, p))
    mc_rows.append(dict(well=w, HCPV_P90_m=q(hc, 10), HCPV_P50_m=q(hc, 50), HCPV_P10_m=q(hc, 90),
                        NETPAY_P90_m=q(npay, 10), NETPAY_P50_m=q(npay, 50), NETPAY_P10_m=q(npay, 90),
                        rank_corr_phi=draws[w][["dphi", "hcpv"]].corr("spearman").iloc[0, 1],
                        rank_corr_m=draws[w][["m", "hcpv"]].corr("spearman").iloc[0, 1],
                        rank_corr_n=draws[w][["n", "hcpv"]].corr("spearman").iloc[0, 1],
                        rank_corr_rw=draws[w][["rwf", "hcpv"]].corr("spearman").iloc[0, 1]))
mc = pd.DataFrame(mc_rows).round(2); mc.to_csv(OUT_DIR / "monte_carlo_hcpv_netpay.csv", index=False)
pd.concat(draws, names=["well"]).to_csv(OUT_DIR / "monte_carlo_draws.csv"); mc

In [ ]:
# Cell 8 — tornado-style sensitivity (rank correlation with HCPV) and distributions
f, ax = plt.subplots(1, 2, figsize=(12, 4.5))
tor = mc.set_index("well")[["rank_corr_phi", "rank_corr_m", "rank_corr_n", "rank_corr_rw"]]
tor.columns = ["porosity shift", "m*", "n*", "Rw"]; tor.plot.barh(ax=ax[0]); ax[0].legend(fontsize=7)
ax[0].axvline(0, color="k", lw=0.8); ax[0].set_xlabel("Spearman rank correlation with HCPV")
for w in WELLS: ax[1].hist(draws[w].hcpv, bins=40, histtype="step", label=w)
ax[1].set_xlabel("HCPV (m) = sum phi(1-Sw)h over net pay"); ax[1].legend(fontsize=7)
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_monte_carlo.png", dpi=150); plt.show()

## Conclusions to carry into Paper 2
1. **Pressure data quality** — of 106 pretests only ~23 give a usable stable build-up; most are dry (tight background sand) or
   supercharged. The usable pressures put F-O1, F-O2 (DST) and F-O8 on a common gas gradient (communication); F-O3/F-O4 read
   high (supercharging). No FWL from pressures.
2. **Capillary pressure** reproduces the SCAL-parameter log Sw and the transition-zone shape in F-O3, F-O4 and F-O6: a thick
   capillary transition in tight rock explains most "high Sw above the GWC".
3. **Sw model choice is the key uncertainty after porosity**: SCAL m* (supported by core Pc) vs water-leg m* (Sw = 1 below the
   GWC). The discrepancy implies either residual gas below the present contact or stress effects on lab data.
4. **F-S1 (and partly F-O8) are not explained by capillarity** — the candidate for perched water / compartmentalisation.
5. **HCPV uncertainty** is dominated by the systematic porosity calibration (±1 p.u.), then m* and n*; Rw is minor.
   No structure maps or GRV are in the data room, so GIIP is not computed here; the per-well HCPV distributions are the input
   once a GRV / area is available.